<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Load, inspect, clean, transform, aggregate, reshape, and join tabular data with explicit validation.</p>
</div>

## Learning objectives

- Understand Series, DataFrame, index, columns, and dtypes.
- Select with `loc` and `iloc` and filter with boolean masks.
- Clean missing, duplicate, text, numeric, and date values.
- Aggregate with groupby and combine tables with validated joins.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Inspect before transforming

A DataFrame is a labeled collection of typed columns. Begin with shape, columns, dtypes, missing counts, uniqueness, and representative rows. Selection with `loc` is label-based; `iloc` is position-based. Avoid chained assignment—select rows and columns in one `.loc[...]` operation.


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "region": ["South", "West", "South", "North"],
    "amount": [1200.0, 850.0, None, 1600.0],
    "status": ["paid", "paid", "cancelled", "paid"],
})

print(orders.info())
print(orders.isna().sum())
print(orders.loc[orders["status"].eq("paid"), ["order_id", "amount"]])


## Cleaning is rule-driven

Missing values are not automatically errors; decide whether to reject, impute, preserve, or flag them from the business meaning. Normalize strings, parse dates with explicit error policy, and convert numeric fields deliberately. Keep an audit column when a transformation changes meaning.


In [ ]:
import pandas as pd

raw = pd.DataFrame({
    "email": [" ASHA@EXAMPLE.COM ", "ravi@example.com", "ravi@example.com"],
    "joined": ["2026-01-10", "bad-date", "bad-date"],
    "score": ["88", "72", "72"],
})

cleaned = raw.assign(
    email=lambda frame: frame["email"].str.strip().str.casefold(),
    joined=lambda frame: pd.to_datetime(frame["joined"], errors="coerce"),
    score=lambda frame: pd.to_numeric(frame["score"], errors="coerce"),
).drop_duplicates()

cleaned["date_valid"] = cleaned["joined"].notna()
print(cleaned)


## Aggregation and joins

`groupby` follows split–apply–combine: split rows into groups, apply aggregations, then combine results. Name aggregations for stable output columns. Joins can multiply rows when keys are non-unique, so use `validate=` and inspect unmatched keys before accepting a result.


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "customer_id": [1, 1, 2, 3],
    "amount": [500, 750, 1200, 400],
})
customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "segment": ["Gold", "Silver", "New"],
})

summary = orders.groupby("customer_id", as_index=False).agg(
    orders=("amount", "size"),
    revenue=("amount", "sum"),
    average_order=("amount", "mean"),
)
result = summary.merge(customers, on="customer_id", how="left", validate="one_to_one")
print(result)


## Worked example: quality-controlled sales summary

Create explicit quality flags before aggregating so the report can state what was excluded.


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "date": ["2026-09-01", "2026-09-01", "bad", "2026-09-02"],
    "region": ["South", "West", "South", "South"],
    "units": [4, 2, 3, -1],
    "price": [800, 1500, 900, 700],
})

prepared = sales.assign(
    date=lambda frame: pd.to_datetime(frame["date"], errors="coerce"),
    valid=lambda frame: frame["date"].notna() & frame["units"].gt(0) & frame["price"].ge(0),
    revenue=lambda frame: frame["units"] * frame["price"],
)

summary = (
    prepared.loc[prepared["valid"]]
    .groupby("region", as_index=False)
    .agg(rows=("revenue", "size"), revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)
print(summary)
print("Rejected rows:", (~prepared["valid"]).sum())


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Build a DataFrame with one duplicate, one missing amount, and one invalid date.
2. Normalize text and parse numeric/date columns.
3. Produce a region summary with count, total, mean, and maximum.
4. Join a unique region lookup with `validate='many_to_one'` and inspect unmatched regions.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
import pandas as pd

transactions = pd.DataFrame({
    "region": [" south ", "WEST", "south", "unknown"],
    "amount": [1000, 800, None, 400],
    "date": ["2026-09-01", "2026-09-02", "bad", "2026-09-04"],
})
lookup = pd.DataFrame({"region": ["South", "West"], "manager": ["Asha", "Ravi"]})

prepared = transactions.assign(
    region=lambda frame: frame["region"].str.strip().str.title(),
    amount=lambda frame: pd.to_numeric(frame["amount"], errors="coerce"),
    date=lambda frame: pd.to_datetime(frame["date"], errors="coerce"),
).drop_duplicates()

summary = prepared.groupby("region", as_index=False).agg(
    rows=("amount", "size"),
    total=("amount", "sum"),
    average=("amount", "mean"),
    maximum=("amount", "max"),
)
enriched = summary.merge(lookup, on="region", how="left", validate="many_to_one")
print(enriched)
print("Unmatched:", enriched.loc[enriched["manager"].isna(), "region"].tolist())


## Knowledge check

**1. What is the first step with a new DataFrame?**

::: {.callout-note collapse="true"}
### Answer
Inspect structure, types, missingness, uniqueness, and sample rows.
:::

**2. Why use join validation?**

::: {.callout-note collapse="true"}
### Answer
It detects unexpected key cardinality and row multiplication.
:::

**3. Why avoid chained assignment?**

::: {.callout-note collapse="true"}
### Answer
It can update a temporary object rather than the intended DataFrame.
:::


## Recap

- Profile first.
- Clean from explicit rules.
- Validate joins and report rejected data.


<div class="lesson-nav">
<a href="11-numpy.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> NumPy for Numerical Computing</a>
<a href="13-visualization.html">Data Visualization and Exploratory Analysis <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
